In [17]:
from langgraph.graph import StateGraph,START,END
from langchain_openai import ChatOpenAI
from typing import TypedDict,Literal,Annotated
from dotenv import load_dotenv
from pydantic import BaseModel, Field
import operator 

In [18]:
load_dotenv()

True

In [19]:
model = ChatOpenAI(model="gpt-4o-mini", temperature=0.9)


In [20]:
class SentimentSchema(BaseModel):
    sentiment: Literal["positive", "negative", "neutral"] = Field(description="The sentiment of the review")

In [21]:
structured_model = model.with_structured_output(SentimentSchema)

In [22]:
class ReviewState(TypedDict):
    review: str
    sentiment_feedback: Literal["positive", "negative", "neutral"]
    diagnosis:dict
    response: str


In [23]:

class DiagnosisSchema(BaseModel):
    issue_type: Literal["UX", "Performance", "Bug", "Support", "Other"] = Field(description='The category of issue mentioned in the review')
    tone: Literal["angry", "frustrated", "disappointed", "calm"] = Field(description='The emotional tone expressed by the user')
    urgency: Literal["low", "medium", "high"] = Field(description='How urgent or critical the issue appears to be')

In [24]:
def analyze_sentiment(state: ReviewState) -> ReviewState:
    sentiment_response = structured_model.invoke(state['review'])
    return {"sentiment_feedback": sentiment_response.sentiment}

In [34]:
def positive_response(state: ReviewState) -> ReviewState:
    prompt=f"""The customer left the following review: {state['review']}.
The sentiment of the review is {state['sentiment_feedback']}."""
    response=model.invoke(prompt).content
    return {"response": response}

In [35]:
def diagnose_issue(state: ReviewState) -> ReviewState:
    structured_diagnosis_model = model.with_structured_output(DiagnosisSchema)
    diagnosis_response = structured_diagnosis_model.invoke(state['review'])
    return {"diagnosis": diagnosis_response.model_dump()}

In [38]:
def negative_response(state: ReviewState):

    diagnosis = state['diagnosis']

    prompt = f"""You are a support assistant.
The user had a '{diagnosis['issue_type']}' issue, sounded '{diagnosis['tone']}', and marked urgency as '{diagnosis['urgency']}'.
Write an empathetic, helpful resolution message.
"""
    response = model.invoke(prompt).content

    return {'response': response}

In [44]:
def check_sentiment(state: ReviewState) -> Literal["positive_response", "diagnose_issue"]:
    if state['sentiment_feedback'] == "positive":
        return "positive_response"
    else:
        return "diagnose_issue"
    

In [ ]:
graph = StateGraph(ReviewState)
graph.add_node("analyze_sentiment", analyze_sentiment)
graph.add_node("diagnose_issue", diagnose_issue)
graph.add_node("positive_response", positive_response)
graph.add_node("negative_response", negative_response)
graph.add_edge(START, "analyze_sentiment")
graph.add_conditional_edges("analyze_sentiment", check_sentiment)
graph.add_edge("positive_response", END)
graph.add_edge("diagnose_issue", "negative_response")
graph.add_edge("negative_response", END)
workflow = graph.compile()

In [49]:
response = workflow.invoke({"review": "The app crashes every time I try to open it. I'm very frustrated!"})
print(response['response'])

Subject: We're Here to Help with Your Bug Issue

Hi [User's Name],

I hope this message finds you well. I understand how frustrating it can be to deal with a bug, especially when it impacts your work. I want to assure you that we are committed to resolving this issue for you as quickly as possible.

Could you please provide me with a bit more detail about the bug you're experiencing? Any specific error messages, steps to reproduce the issue, or screenshots would be incredibly helpful. This information will allow us to diagnose the problem more effectively and get you back on track.

Thank you for your patience as we work to resolve this. We're here for you, and I appreciate your understanding.

Best regards,

[Your Name]  
[Your Title]  
[Your Contact Information]  
